In [13]:
import os
import sys
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/OZON')
os.chdir(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / "data"
SUB_DIR = PROJECT_DIR / "submissions"
DATA_DIR.mkdir(parents=True, exist_ok=True)
SUB_DIR.mkdir(parents=True, exist_ok=True)

print(f"Рабочая директория: {PROJECT_DIR}")
print(f"Исходный train.parquet найден: {(PROJECT_DIR / 'train.parquet').exists()}")
print(f"Файлы в директории: {os.listdir(PROJECT_DIR)}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Рабочая директория: /content/drive/MyDrive/OZON
Исходный train.parquet найден: True
Файлы в директории: ['train.parquet', 'README.md', 'train.py', 'time_split.py', 'predict.py', 'make_synthetic_data.py', 'features.py', 'data_loading.py', 'btyd_features.py', 'build_dataset.py', 'config.py', 'data', 'submissions', '__pycache__', 'uploads']


In [25]:
import os
import warnings
from pathlib import Path
from google.colab import drive
import numpy as np
import pandas as pd
from sklearn.preprocessing import QuantileTransformer
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/OZON')
os.chdir(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / 'data'
SUB_DIR = PROJECT_DIR / 'submissions'
SUB_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


class FastTabularDataset(Dataset):
  def __init__(self, X, y=None):
    self.X = torch.tensor(X, dtype=torch.float32)
    self.y = (
        torch.tensor(y, dtype=torch.float32).unsqueeze(1)
        if y is not None
        else None
    )

  def __len__(self):
    return len(self.X)

  def __getitem__(self, idx):
    return (self.X[idx], self.y[idx]) if self.y is not None else self.X[idx]


class ResNetBlock(nn.Module):

  def __init__(self, dim, dropout=0.1):
    super().__init__()
    self.block = nn.Sequential(
        nn.BatchNorm1d(dim),
        nn.SiLU(),
        nn.Dropout(dropout),
        nn.Linear(dim, dim),
        nn.BatchNorm1d(dim),
        nn.SiLU(),
        nn.Dropout(dropout),
        nn.Linear(dim, dim),
    )

  def forward(self, x):
    return x + self.block(x)


class FastTabularResNet(nn.Module):

  def __init__(self, input_dim, hidden_dim=512, num_blocks=3, dropout=0.1):
    super().__init__()
    self.input_layer = nn.Sequential(
        nn.Linear(input_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.SiLU()
    )
    self.blocks = nn.ModuleList(
        [ResNetBlock(hidden_dim, dropout) for _ in range(num_blocks)]
    )
    self.head = nn.Linear(hidden_dim, 1)

  def forward(self, x):
    x = self.input_layer(x)
    for block in self.blocks:
      x = block(x)
    return self.head(x)


def rmsle(y_true, y_pred):
  return float(
      np.sqrt(
          np.mean(
              (np.log1p(np.clip(y_pred, 0, None)) - np.log1p(np.clip(y_true, 0, None)))
              ** 2
          )
      )
  )


fold_files = sorted(list(DATA_DIR.glob('fold_*.parquet')))
test_df = pd.read_parquet(DATA_DIR / 'test_features.parquet')
folds = [pd.read_parquet(f) for f in fold_files]

exclude_cols = {
    'user_id',
    'target',
    'target_log',
    'fold',
    'cutoff_date',
    'first_order_dt',
    'last_order_dt',
}
feature_cols = [c for c in folds[0].columns if c not in exclude_cols]

BATCH_SIZE = 8192
EPOCHS = 35
LR = 3e-3
PATIENCE = 5

oof_preds, targets, test_preds_list = [], [], []
scaler_cuda = torch.cuda.amp.GradScaler()

for val_fold_idx, val_df in tqdm(
    enumerate(folds), total=len(folds), desc='Folds'
):
  train_df = pd.concat(
      [folds[i] for i in range(len(folds)) if i != val_fold_idx],
      ignore_index=True,
  )

  scaler = QuantileTransformer(
      output_distribution='normal', random_state=42, n_quantiles=1000
  )
  X_train = scaler.fit_transform(train_df[feature_cols].fillna(0).values)
  X_val = scaler.transform(val_df[feature_cols].fillna(0).values)
  X_test = scaler.transform(test_df[feature_cols].fillna(0).values)

  train_loader = DataLoader(
      FastTabularDataset(X_train, train_df['target_log'].values),
      batch_size=BATCH_SIZE,
      shuffle=True,
      num_workers=2,
      pin_memory=True,
  )
  val_loader = DataLoader(
      FastTabularDataset(X_val, val_df['target_log'].values),
      batch_size=BATCH_SIZE,
      shuffle=False,
      num_workers=2,
      pin_memory=True,
  )
  test_loader = DataLoader(
      FastTabularDataset(X_test),
      batch_size=BATCH_SIZE,
      shuffle=False,
      num_workers=2,
      pin_memory=True,
  )

  model = FastTabularResNet(len(feature_cols)).to(device)
  criterion = nn.MSELoss()
  optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
  scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
      optimizer, mode='min', factor=0.5, patience=2
  )

  best_loss, best_val_preds, patience_counter = float('inf'), None, 0

  for epoch in range(EPOCHS):
    model.train()
    for bx, by in train_loader:
      bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
      optimizer.zero_grad()
      with torch.cuda.amp.autocast():
        loss = criterion(model(bx), by)
      scaler_cuda.scale(loss).backward()
      scaler_cuda.step(optimizer)
      scaler_cuda.update()

    model.eval()
    val_loss, current_val_preds = 0.0, []
    with torch.no_grad():
      for bx, by in val_loader:
        bx, by = bx.to(device, non_blocking=True), by.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
          out = model(bx)
          val_loss += criterion(out, by).item() * len(bx)
        current_val_preds.append(out.cpu().numpy())

    val_loss /= len(val_df)
    scheduler.step(val_loss)

    if val_loss < best_loss:
      best_loss = val_loss
      best_val_preds = np.vstack(current_val_preds).ravel()
      torch.save(
          model.state_dict(), DATA_DIR / f'nn_model_fold_{val_fold_idx}.pth'
      )
      patience_counter = 0
    else:
      patience_counter += 1
      if patience_counter >= PATIENCE:
        break

  val_preds_gmv = np.expm1(np.clip(best_val_preds, 0, None))
  oof_preds.extend(val_preds_gmv)
  targets.extend(val_df['target'].values)

  model.load_state_dict(
      torch.load(DATA_DIR / f'nn_model_fold_{val_fold_idx}.pth')
  )
  model.eval()
  t_preds = []
  with torch.no_grad():
    for bx in test_loader:
      bx = bx.to(device, non_blocking=True)
      with torch.cuda.amp.autocast():
        t_preds.append(model(bx).cpu().numpy())
  test_preds_list.append(
      np.expm1(np.clip(np.vstack(t_preds).ravel(), 0, None))
  )

sub_nn = pd.DataFrame(
    {'user_id': test_df['user_id'], 'predict': np.mean(test_preds_list, axis=0)}
)
sub_nn.to_csv(SUB_DIR / 'submission_nn.csv', index=False)
print(f'Итоговый OOF RMSLE (NN): {rmsle(np.array(targets), np.array(oof_preds)):.5f}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Используем устройство: cuda
Признаков: 133, Найдено фолдов: 6

--- Обучение НС на фолде 0 ---
Fold 0 Best RMSE (Log): 1.69653 | RMSLE (GMV): 1.69653

--- Обучение НС на фолде 1 ---
Fold 1 Best RMSE (Log): 1.70682 | RMSLE (GMV): 1.70682

--- Обучение НС на фолде 2 ---
Fold 2 Best RMSE (Log): 1.73050 | RMSLE (GMV): 1.73050

--- Обучение НС на фолде 3 ---
Fold 3 Best RMSE (Log): 1.75983 | RMSLE (GMV): 1.75983

--- Обучение НС на фолде 4 ---
Fold 4 Best RMSE (Log): 1.72504 | RMSLE (GMV): 1.72504

--- Обучение НС на фолде 5 ---
Fold 5 Best RMSE (Log): 1.67745 | RMSLE (GMV): 1.67745

ИТОГОВЫЙ OOF RMSLE (Neural Network): 1.71623
Сабмит НС сохранен в: /content/drive/MyDrive/OZON/submissions/submission_nn.csv


In [26]:
from pathlib import Path
import pandas as pd

SUB_DIR = Path('/content/drive/MyDrive/OZON/submissions')

lgb_path = SUB_DIR / 'submission_log_target.csv'
nn_path = SUB_DIR / 'submission_nn.csv'
cb_path = SUB_DIR / 'submission_catboost.csv'

lgb_sub = pd.read_csv(lgb_path) if lgb_path.exists() else None
nn_sub = pd.read_csv(nn_path) if nn_path.exists() else None
cb_sub = pd.read_csv(cb_path) if cb_path.exists() else None

for df in [lgb_sub, nn_sub, cb_sub]:
  if df is not None and 'target' in df.columns:
    df.rename(columns={'target': 'predict'}, inplace=True)

final_sub = lgb_sub.copy()

if cb_sub is not None:
  print('Создаём 3-компонентный бленд...')
  final_sub['predict'] = (
      0.50 * lgb_sub['predict']
      + 0.35 * cb_sub['predict']
      + 0.15 * nn_sub['predict']
  )
else:
  print('Создаём 2-компонентный бленд...')
  final_sub['predict'] = 0.85 * lgb_sub['predict'] + 0.15 * nn_sub['predict']

out_path = SUB_DIR / 'submission_blend_final.csv'
final_sub.to_csv(out_path, index=False)
print(f'Финальный сабмит сохранён в: {out_path}')

 Готово! Ансамбль успешно сформирован:
Файл: /content/drive/MyDrive/OZON/submissions/submission_blend_lgb_nn.csv

Первые строки сабмита:
   user_id     predict
0        2    2.018496
1        7   65.082647
2       15    8.710345
3       18  117.561835
4       23    0.458226
